In [1]:
import pandas as pd
import Levenshtein
from scipy.stats import entropy

In [31]:
# 1. Load the dataset
df = pd.read_csv('Data/ostschweiz_mapping_results.csv')
df['IPA_Dialekt'] = df['IPA_Dialekt'].fillna('').astype(str)
df['Hochdeutsch_Zuordnung'] = df['Hochdeutsch_Zuordnung'].fillna('').astype(str)

print("--- 0. FILTERING CO-OCCURRENCE NOISE ---")
# 1. First, calculate the normalized distance for ALL pairs
def calc_norm_distance(row):
    ipa = row['IPA_Dialekt']
    hd = row['Hochdeutsch_Zuordnung']
    max_len = max(len(ipa), len(hd))
    if max_len == 0: return 0.0
    return Levenshtein.distance(ipa, hd) / max_len

df['Norm_Distance'] = df.apply(calc_norm_distance, axis=1)

# 2. Apply our Statistical Filter (at least 15% confidence)
ipa_totals = df.groupby('IPA_Dialekt')['Gemeinsame_Treffer'].transform('sum')
df['Match_Confidence'] = df['Gemeinsame_Treffer'] / ipa_totals
stat_clean = df[df['Match_Confidence'] > 0.15]

# 3. Apply a Phonetic Sanity Check (NEW)
# If the IPA and HD words are completely different (e.g., > 80% different characters), 
# they are almost certainly a co-occurrence false positive (like 'ʊnt' matching 'die').
clean_df = stat_clean[stat_clean['Norm_Distance'] < 0.80].copy()

print(f"Reduced raw pairs from {len(df)} to {len(clean_df)} strong, phonetically sound pairs.\n")

--- 0. FILTERING CO-OCCURRENCE NOISE ---
Reduced raw pairs from 1176 to 141 strong, phonetically sound pairs.



In [33]:
print("--- 1. VARIANTS PER STANDARD GERMAN WORD ---")
# Group by Standard German and get all unique IPA pronunciations as a list
variant_lists = clean_df.groupby('Hochdeutsch_Zuordnung')['IPA_Dialekt'].unique()

# Count how many variants each word has
variant_counts = variant_lists.apply(len)

always_same = (variant_counts == 1).sum()
multiple_variants = (variant_counts > 1).sum()

print(f"Words always spoken the same (1 variant): {always_same}")
print(f"Words with multiple IPA variants: {multiple_variants}")

print("\nTop 5 words with the MOST variants and their pronunciations:")
# Find the 5 words with the highest number of variants
top_5_words = variant_counts.sort_values(ascending=False).head(20).index

# Print the word, the count, and the actual IPA variants
for word in top_5_words:
    variants = variant_lists[word]
    print(f" - '{word}' ({len(variants)} variants): {', '.join(variants)}")

--- 1. VARIANTS PER STANDARD GERMAN WORD ---
Words always spoken the same (1 variant): 46
Words with multiple IPA variants: 21

Top 5 words with the MOST variants and their pronunciations:
 - 'die' (16 variants): diː, diːɾ, deːm, iː, dɛm, ziːn, də, deːɾ, dɛt, daɪn, fiːɾ, miːt, dɛnt, duː, dɑːɾ, diːn
 - 'der' (12 variants): deː, fɔr, dɛn, də, diːɾ, dɛ, dɛt, fɔːr, deːm, dɛm, vɔːr, foːr
 - 'das' (8 variants): das, dɑːs, dɑs, dɑz, tas, daɪs, ɑs, tɛs
 - 'nicht' (5 variants): nɛt, nøːt, nøt, nɪt, nɪçt
 - 'und' (5 variants): ʊnt, ɔn, ʊn, ɛnt, ʊntə
 - 'ist' (5 variants): ɪst, nɛt, iːʃ, zɪçt, ɪnst
 - 'mit' (5 variants): mɪt, miːt, miː, miːd, mɪç
 - 'wir' (4 variants): miːɾ, miː, meːr, miːr
 - 'auf' (4 variants): ɔf, uːf, ʊf, aʊf
 - 'sie' (4 variants): ziː, tsiː, si, tsi
 - 'für' (4 variants): fyːɾ, fɛɾ, fiːɾ, fyːr
 - 'sind' (3 variants): zɪnt, ɪnt, ziːn
 - 'dem' (3 variants): deːm, dɛm, ɛm
 - 'am' (3 variants): am, ɑːm, ɑm
 - 'im' (2 variants): ɪm, diːm
 - 'hat' (2 variants): haɪt, hɛlt
 - 'doch

In [34]:
print("\n--- 2. AMBIGUOUS IPA WORDS (HOMOPHONES) ---")
# Group by IPA and get all unique Standard German translations as a list
hd_lists = clean_df.groupby('IPA_Dialekt')['Hochdeutsch_Zuordnung'].unique()

# Count how many Standard German words each IPA string maps to
hd_counts = hd_lists.apply(len)

ambiguous_ipa = hd_counts[hd_counts > 1]
print(f"Number of IPA strings mapping to multiple Standard German words: {len(ambiguous_ipa)}")

if len(ambiguous_ipa) > 0:
    print("\nTop 5 most ambiguous IPA strings and their Standard German meanings:")
    # Find the 5 IPA strings with the highest number of different meanings
    top_5_ipa = ambiguous_ipa.sort_values(ascending=False).head(30).index
    
    # Print the IPA, the count, and the actual German words
    for ipa in top_5_ipa:
        meanings = hd_lists[ipa]
        print(f" - [{ipa}] ({len(meanings)} meanings): {', '.join(meanings)}")


--- 2. AMBIGUOUS IPA WORDS (HOMOPHONES) ---
Number of IPA strings mapping to multiple Standard German words: 14

Top 5 most ambiguous IPA strings and their Standard German meanings:
 - [deːm] (3 meanings): die, dem, der
 - [diːɾ] (3 meanings): die, der, diese
 - [dɛm] (3 meanings): die, dem, der
 - [fiːɾ] (3 meanings): für, vier, die
 - [andərə] (2 meanings): andere, anderen
 - [dɑs] (2 meanings): das, dass
 - [də] (2 meanings): der, die
 - [dɛt] (2 meanings): der, die
 - [fiːl] (2 meanings): viel, viele
 - [miː] (2 meanings): wir, mit
 - [miːt] (2 meanings): mit, die
 - [mɪç] (2 meanings): mich, mit
 - [nɛt] (2 meanings): nicht, ist
 - [ziːn] (2 meanings): die, sind


In [15]:
print("\n--- 3. MOST DIFFERENT WORDS (LEVENSHTEIN) ---")
def calc_norm_distance(row):
    ipa = row['IPA_Dialekt']
    hd = row['Hochdeutsch_Zuordnung']
    max_len = max(len(ipa), len(hd))
    if max_len == 0: return 0.0
    return Levenshtein.distance(ipa, hd) / max_len

clean_df['Norm_Distance'] = clean_df.apply(calc_norm_distance, axis=1)

# Look at words with high confidence to ensure we are comparing real translations
most_different = clean_df[clean_df['Gemeinsame_Treffer'] > 10].sort_values(by='Norm_Distance', ascending=False)
print("Top 5 biggest orthographic/phonetic deviations:")
print(most_different[['IPA_Dialekt', 'Hochdeutsch_Zuordnung', 'Norm_Distance', 'Gemeinsame_Treffer']].head(20))


--- 3. MOST DIFFERENT WORDS (LEVENSHTEIN) ---
Top 5 biggest orthographic/phonetic deviations:
    IPA_Dialekt Hochdeutsch_Zuordnung  Norm_Distance  Gemeinsame_Treffer
119        deːm                   die       0.750000                  19
39         zɪnt                  sind       0.750000                  35
37           aʊ                  auch       0.750000                  35
171        viːɾ                   die       0.750000                  16
189        viːɾ                   wie       0.750000                  15
155        viːɾ                  wird       0.750000                  17
58         tsʊm                   zum       0.750000                  30
248        diːɾ                   der       0.750000                  13
66         miːɾ                   wir       0.750000                  28
48          nɔx                  nach       0.750000                  33
133         dɔx                  doch       0.750000                  18
265          ɛn              

In [16]:
print("\n--- 4. PRONUNCIATION ENTROPY ---")
def calculate_entropy(group):
    total_treffer = group['Gemeinsame_Treffer'].sum()
    if total_treffer == 0: return 0.0
    probabilities = group['Gemeinsame_Treffer'] / total_treffer
    return entropy(probabilities, base=2)

entropy_df = clean_df.groupby('Hochdeutsch_Zuordnung').apply(calculate_entropy).reset_index(name='Entropy')
word_counts = clean_df.groupby('Hochdeutsch_Zuordnung')['Gemeinsame_Treffer'].sum()
valid_words = word_counts[word_counts > 20].index

high_entropy = entropy_df[entropy_df['Hochdeutsch_Zuordnung'].isin(valid_words)].sort_values(by='Entropy', ascending=False)
print("Top 5 Standard German words with the highest pronunciation unpredictability:")
print(high_entropy.head(5))


--- 4. PRONUNCIATION ENTROPY ---
Top 5 Standard German words with the highest pronunciation unpredictability:
   Hochdeutsch_Zuordnung   Entropy
24                   die  3.646896
22                   der  3.413332
18                   das  3.220430
41                   ist  2.709718
40                    in  2.492922


C:\Users\chris\AppData\Local\Temp\ipykernel_11000\758554440.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  entropy_df = clean_df.groupby('Hochdeutsch_Zuordnung').apply(calculate_entropy).reset_index(name='Entropy')
